# 03 - Model Training & Comparison

This notebook trains and evaluates machine learning classifiers for distinguishing **Primary Myelofibrosis (PMF)** from control samples using two molecular data types:

- Gene expression
- miRNA expression

The dataset contains substantially more molecular features than samples, making dimensionality reduction and leakage-aware model evaluation important considerations.

### Objectives

The analysis aims to:

1. Prepare aligned feature matrices and class labels.
2. Characterize dataset dimensions and class balance.
3. Evaluate models using **stratified 5-fold cross-validation**.
4. Perform **feature selection within each cross-validation fold** to prevent information leakage.
5. Apply model-appropriate preprocessing.
6. Compare Logistic Regression and Random Forest classifiers.
7. Generate out-of-fold predictions for downstream evaluation.
8. Save reproducible performance metrics and predictions.

### Models

Two complementary classifiers are evaluated:

- **Logistic Regression** — a linear model that provides a relatively interpretable baseline for high-dimensional expression data.
- **Random Forest** — a nonlinear ensemble model capable of capturing interactions and nonlinear relationships between selected features.

### Evaluation Strategy

Because the dataset contains only 73 samples, a single train/test split would produce a relatively small and potentially unstable test set.

Instead, model performance is estimated using **stratified 5-fold cross-validation**. Stratification preserves the PMF/control class ratio across folds.

Feature selection is performed inside the scikit-learn pipelines, ensuring that feature rankings are calculated independently within each training fold rather than using information from the validation fold.

### Import Required Libraries

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)
from sklearn.base import clone

# Add project root to the Python path so that project-level
sys.path.append(os.path.abspath(".."))

# Import custom function for model evaluation
from Src.Model_Utility import evaluate_model, create_pipelines

### Load Aligned Expression and Metadata

The processed expression matrices and corresponding metadata generated during the preprocessing stage are loaded for model development.

Each expression matrix is structured as:

- **Rows:** samples
- **Columns:** molecular features

The corresponding metadata contains the binary classification label:

- `1` → PMF
- `0` → Control

Sample identifiers were aligned during preprocessing. Alignment is checked again here before model training to ensure that expression values and labels correspond to the same samples.

In [47]:
# Load aligned gene expression dataset
gene_df = pd.read_csv("../Data/Processed/gene_expression_aligned.csv", index_col=0)

# Load gene expression metadata
meta_gene = pd.read_csv("../Data/Processed/gene_metadata.csv", index_col=0)

# Load aligned miRNA expression dataset
mirna_df = pd.read_csv("../Data/Processed/mirna_expression_aligned.csv", index_col=0)

# Load miRNA expression metadata
meta_mirna = pd.read_csv("../Data/Processed/mirna_metadata.csv", index_col=0)

### Prepare Features and Labels

The expression matrices are converted into feature matrices (`X`) and target vectors (`y`) for scikit-learn.

For both molecular data types:

- `X` contains expression measurements.
- `y` contains the binary disease classification.

The sample order is verified before converting the data to NumPy arrays. This prevents an incorrect correspondence between expression profiles and their labels.

In [48]:
# Verify that expression samples and metadata are in identical order.
assert list(gene_df.index) == list(meta_gene.index), \
    "Gene expression samples are not aligned with gene metadata."

assert list(mirna_df.index) == list(meta_mirna.index), \
    "miRNA expression samples are not aligned with miRNA metadata."

# Convert expression matrices to feature arrays.
X_gene = gene_df.values
X_mirna = mirna_df.values

# Extract binary classification labels.
y_gene = meta_gene["label"].values
y_mirna = meta_mirna["label"].values

### Dataset Dimensions and Class Balance

The original processed datasets contain substantially more features than samples:

- **Gene expression:** 73 samples × 49,386 features
- **miRNA expression:** 73 samples × 20,212 features

This high-dimensional setting motivates the use of supervised feature selection before model fitting.

The class distribution is also inspected to confirm the number of PMF and control samples available for stratified cross-validation.

In [49]:
print("Gene expression shape:", X_gene.shape)
print("miRNA expression shape:", X_mirna.shape)

print("\nGene expression class distribution:")
print(pd.Series(y_gene).value_counts())

print("\nmiRNA expression class distribution:")
print(pd.Series(y_mirna).value_counts())

Gene expression shape: (73, 49386)
miRNA expression shape: (73, 20212)

Gene expression class distribution:
1    42
0    31
Name: count, dtype: int64

miRNA expression class distribution:
1    42
0    31
Name: count, dtype: int64


### Define Cross-Validation and Feature-Selection Values

Use stratified 5-fold cross-validation to preserve the PMF/control class distribution across folds. Multiple feature counts are evaluated to assess how model performance changes with the number of selected features

In [50]:
# Use stratified folds to preserve the PMF/control class distribution.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Candidate numbers of features to evaluate.
k_values = [10, 25, 50, 100, 250, 500, 1000]

### Evaluate Model Performance

Evaluate Logistic Regression and Random Forest across the candidate feature counts for both gene expression and miRNA datasets. The results are stored for comparison and selection of an appropriate feature-set size for each model and dataset.

In [51]:
# Evaluate each feature-set size using the same cross-validation strategy.
# The preferred k will be determined separately for each model and dataset.

# Store the results for each dataset/model/k combination.
k_results = []

datasets = {
    "Gene": (X_gene, y_gene),
    "miRNA": (X_mirna, y_mirna)
}

for data_type, (X, y) in datasets.items():

    for k in k_values:

        # Create pipelines using the current number of selected features.
        logreg_pipeline, rf_pipeline = create_pipelines(k)

        # Evaluate Logistic Regression.
        logreg_result = evaluate_model(
            logreg_pipeline,
            X,
            y,
            "Logistic Regression",
            data_type,
            cv
        )

        logreg_result["k_features"] = k
        k_results.append(logreg_result)

        # Evaluate Random Forest.
        rf_result = evaluate_model(
            rf_pipeline,
            X,
            y,
            "Random Forest",
            data_type,
            cv
        )

        rf_result["k_features"] = k
        k_results.append(rf_result)

# Combine all feature-selection results into a single table.
k_results_df = pd.DataFrame(k_results)

k_results_df

,Dataset,Model,ROC_AUC_mean,ROC_AUC_std,Accuracy_mean,Accuracy_std,Precision_mean,Recall_mean,F1_mean,k_features
0,Gene,Logistic Regression,1.000000,0.000000,0.972381,0.033860,0.977778,0.977778,0.976471,10
1,Gene,Random Forest,1.000000,0.000000,0.973333,0.032660,0.980000,0.977778,0.977709,10
2,Gene,Logistic Regression,1.000000,0.000000,0.986667,0.026667,1.000000,0.977778,0.988235,25
3,Gene,Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,25
4,Gene,Logistic Regression,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,50
5,Gene,Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,50
6,Gene,Logistic Regression,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,100
7,Gene,Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,100
8,Gene,Logistic Regression,1.000000,0.000000,0.985714,0.028571,1.000000,0.975000,0.986667,250
9,Gene,Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,250


### Select the Parsimonious Feature-Set Size

Select a feature-set size separately for each dataset and model. Candidate values are considered equivalent when their ROC-AUC and F1 scores are within a defined tolerance of the best observed performance. Among these candidates, the smallest feature set is preferred, with lower ROC-AUC variability used as a secondary criterion.

In [52]:
# Define how close a model must be to the best observed performance
# before it is considered practically equivalent.
auc_tolerance = 0.01
f1_tolerance = 0.01


# Store the selected k for each dataset/model combination.
best_k_results = []

# Evaluate each dataset/model combination independently.
for (dataset, model), group in k_results_df.groupby(["Dataset", "Model"]):

    # Find the best ROC-AUC and F1 observed across the tested
    # feature counts for this dataset/model combination.
    best_auc = group["ROC_AUC_mean"].max()
    best_f1 = group["F1_mean"].max()

    # Keep feature counts whose ROC-AUC is close to the best result.
    auc_candidates = group[
        group["ROC_AUC_mean"] >= best_auc - auc_tolerance
    ]

    # Of those candidates, keep only models whose F1 score is also
    # close to the best observed F1.
    candidates = auc_candidates[
        auc_candidates["F1_mean"] >= best_f1 - f1_tolerance
    ]

    # Sort by:
    # 1. Smallest number of features (prefer simpler models)
    # 2. Lower ROC-AUC variability
    # 3. Lower F1 variability if available
    candidates = candidates.sort_values(
        ["k_features", "ROC_AUC_std"],
        ascending=[True, True]
    )

    # Select the most parsimonious candidate.
    best_k_results.append(
        candidates.iloc[0]
    )


# Combine the selected rows into a summary DataFrame.
best_k_df = pd.DataFrame(best_k_results).reset_index(drop=True)

best_k_df

,Dataset,Model,ROC_AUC_mean,ROC_AUC_std,Accuracy_mean,Accuracy_std,Precision_mean,Recall_mean,F1_mean,k_features
0,Gene,Logistic Regression,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.00000,50
1,Gene,Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.00000,25
2,miRNA,Logistic Regression,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.00000,1000
3,miRNA,Random Forest,0.985185,0.018144,0.919048,0.077606,0.952778,0.908333,0.92915,250


### Define Final Model Pipelines

The selected feature-set size is applied separately to each dataset and classifier.

In [53]:
# Retrieve the selected k for each model and dataset combination.
best_k = {
    row["Dataset"] + "_" + row["Model"]: row["k_features"]
    for _, row in best_k_df.iterrows()
}

gene_logreg_k = best_k["Gene_Logistic Regression"]
gene_rf_k = best_k["Gene_Random Forest"]

mirna_logreg_k = best_k["miRNA_Logistic Regression"]
mirna_rf_k = best_k["miRNA_Random Forest"]

# Create the final pipelines using the selected feature-set sizes.

gene_logreg_pipeline, _ = create_pipelines(gene_logreg_k)
_, gene_rf_pipeline = create_pipelines(gene_rf_k)

mirna_logreg_pipeline, _ = create_pipelines(mirna_logreg_k)
_, mirna_rf_pipeline = create_pipelines(mirna_rf_k)

### Extract Fold-Level Feature Selection and Importance

Feature importance is extracted from each cross-validation fold using the selected feature set for that fold. 

Recording the selected features and their corresponding importance values across folds allows feature stability and model interpretation to be assessed without relying on a single fitted model.

In [54]:
# Store the selected k for each dataset/model combination.
feature_importance_results = []

models = {
    "LogReg_Gene": (gene_logreg_pipeline, X_gene, y_gene, gene_df.columns),
    "RF_Gene": (gene_rf_pipeline, X_gene, y_gene, gene_df.columns),
    "LogReg_miRNA": (mirna_logreg_pipeline, X_mirna, y_mirna, mirna_df.columns),
    "RF_miRNA": (mirna_rf_pipeline, X_mirna, y_mirna, mirna_df.columns)
}

for model_name, (pipeline, X, y, feature_names) in models.items():

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X, y),
        start=1
    ):

        model = clone(pipeline)

        X_train = X[train_idx]
        y_train = y[train_idx]

        # Fit the complete pipeline on the training fold.
        model.fit(X_train, y_train)

        # Extract the fitted feature-selection step.
        selector = model.named_steps["feature_selection"]

        # Identify the features selected within this fold.
        selected_mask = selector.get_support()
        selected_features = np.array(feature_names)[selected_mask]

        # Extract model-specific feature importance.
        classifier = model.named_steps["classifier"]

        if hasattr(classifier, "coef_"):
            importance = classifier.coef_[0]

        elif hasattr(classifier, "feature_importances_"):
            importance = classifier.feature_importances_

        else:
            raise ValueError(
                f"Model {model_name} does not provide feature importance."
            )

        # Store the selected features and their importance for this fold.
        for feature, value in zip(selected_features, importance):

            feature_importance_results.append({
                "Model": model_name,
                "Fold": fold,
                "Feature": feature,
                "Importance": value
            })

feature_importance_df = pd.DataFrame(feature_importance_results)

feature_importance_df.head()

,Model,Fold,Feature,Importance
0,LogReg_Gene,1,11715380_s_at,0.139162
1,LogReg_Gene,1,11715670_a_at,0.194029
2,LogReg_Gene,1,11715671_x_at,0.189578
3,LogReg_Gene,1,11715946_a_at,0.146438
4,LogReg_Gene,1,11716395_a_at,0.125511


### Cross-Validated Model Performance

Each model is evaluated independently on the gene expression and miRNA datasets using the same stratified cross-validation strategy.

The following metrics are recorded:

- **ROC-AUC** — ability to discriminate between PMF and control samples across classification thresholds.
- **Accuracy** — proportion of correctly classified samples.
- **Precision** — proportion of predicted PMF samples that are actually PMF.
- **Recall** — proportion of PMF samples correctly identified.
- **F1 score** — harmonic mean of precision and recall.

For each metric, the mean performance across folds is reported. Standard deviation is also reported for ROC-AUC and accuracy to describe fold-to-fold variability.

In [55]:
results = []

# Evaluate the final selected Logistic Regression and Random Forest
# pipelines for the gene expression dataset.

results.append(
    evaluate_model(
        gene_logreg_pipeline,
        X_gene,
        y_gene,
        "Logistic Regression",
        "Gene",
        cv
    )
)

results.append(
    evaluate_model(
        gene_rf_pipeline,
        X_gene,
        y_gene,
        "Random Forest",
        "Gene",
        cv
    )
)

# Evaluate the final selected pipelines for the miRNA dataset.

results.append(
    evaluate_model(
        mirna_logreg_pipeline,
        X_mirna,
        y_mirna,
        "Logistic Regression",
        "miRNA",
        cv
    )
)

results.append(
    evaluate_model(
        mirna_rf_pipeline,
        X_mirna,
        y_mirna,
        "Random Forest",
        "miRNA",
        cv
    )
)

results = pd.DataFrame(results)

results

,Dataset,Model,ROC_AUC_mean,ROC_AUC_std,Accuracy_mean,Accuracy_std,Precision_mean,Recall_mean,F1_mean
0,Gene,Logistic Regression,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.00000
1,Gene,Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.00000
2,miRNA,Logistic Regression,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.00000
3,miRNA,Random Forest,0.985185,0.018144,0.919048,0.077606,0.952778,0.908333,0.92915


Add the selected feature-set size to the final model performance results.

In [56]:
# Record the selected number of features for each dataset/model combination.
results["k_features"] = [
    gene_logreg_k,
    gene_rf_k,
    mirna_logreg_k,
    mirna_rf_k
]

results

,Dataset,Model,ROC_AUC_mean,ROC_AUC_std,Accuracy_mean,Accuracy_std,Precision_mean,Recall_mean,F1_mean,k_features
0,Gene,Logistic Regression,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.00000,50
1,Gene,Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.00000,25
2,miRNA,Logistic Regression,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.00000,1000
3,miRNA,Random Forest,0.985185,0.018144,0.919048,0.077606,0.952778,0.908333,0.92915,250


### Generate Out-of-Fold Predictions

Out-of-fold predictions are generated using the final pipelines with their selected feature-set sizes.

Each sample is predicted using a model that was not trained on that sample.
Predicted class probabilities and binary classifications are retained.
These predictions provide a consistent basis for downstream ROC curves and confusion matrices in Notebook 4.

In [57]:
# Generate out-of-fold predictions using the final selected pipelines.
oof_predictions = {}

models = {
    "LogReg_Gene": (gene_logreg_pipeline, X_gene, y_gene),
    "RF_Gene": (gene_rf_pipeline, X_gene, y_gene),
    "LogReg_miRNA": (mirna_logreg_pipeline, X_mirna, y_mirna),
    "RF_miRNA": (mirna_rf_pipeline, X_mirna, y_mirna)
}

for name, (model, X, y) in models.items():

    # Predict PMF probability for each held-out sample.
    probabilities = cross_val_predict(
        model,
        X,
        y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    predictions = (probabilities >= 0.5).astype(int)

    oof_predictions[name] = {
        "y_true": y,
        "y_pred": predictions,
        "y_prob": probabilities
    }

### Save Model Results

The cross-validation summary,out-of-fold predictions, and feature importance summary are saved to the `Results/` directory.

In [58]:
# Save results to CSV for downstream analysis
os.makedirs("../Results", exist_ok=True)
os.makedirs("../Results/Tables", exist_ok=True)
os.makedirs("../Results/Tables/Cross_Validation", exist_ok=True)
results.to_csv(
    "../Results/Tables/Cross_Validation/Cross_Validation_Results.csv",
    index=False
)

# Save predictions for downstream evaluation.
os.makedirs("../Results/Tables/OOF_Predictions", exist_ok=True)
for name, pred in oof_predictions.items():

    pd.DataFrame({
        "y_true": pred["y_true"],
        "y_pred": pred["y_pred"],
        "y_prob": pred["y_prob"]
    }).to_csv(
        f"../Results/Tables/OOF_Predictions/{name}_oof_predictions.csv",
        index=False
    )

# Save feature importance for downstream evaluation.
os.makedirs("../Results/Tables/Feature_Importance", exist_ok=True)
feature_importance_df.to_csv(
    "../Results/Tables/Feature_Importance/Feature_Importance_By_Fold.csv",
    index=False
)

### Summary

This notebook established the machine-learning workflow for distinguishing Primary Myelofibrosis (PMF) from control samples using gene-expression and miRNA-expression data.

The analysis used stratified 5-fold cross-validation, with feature selection performed inside the modeling pipelines to prevent information leakage between training and validation folds. Logistic Regression and Random Forest classifiers were evaluated across multiple feature-set sizes, and a parsimonious feature count was selected separately for each dataset and model.

The final cross-validated results were:

- Gene expression + Logistic Regression: ROC-AUC = 1.00, F1 = 1.00, using 50 features
- Gene expression + Random Forest: ROC-AUC = 1.00, F1 = 1.00, using 25 features
- miRNA + Logistic Regression: ROC-AUC = 1.00, F1 = 1.00, using 1,000 features
- miRNA + Random Forest: ROC-AUC = 0.985, F1 = 0.929, using 250 features

The results indicate very strong cross-validated discrimination between PMF and controls, particularly for the gene-expression models and the miRNA Logistic Regression model. However, given the small sample size (73 samples) and the extremely high dimensionality of the original expression matrices, these performance estimates should be interpreted cautiously and validated independently before drawing biological or clinical conclusions.

Out-of-fold predictions were generated for all four final models, ensuring that each sample was evaluated by a model that did not use that sample during training. Fold-level feature-selection and importance information was also retained to support downstream assessment of model interpretation and feature stability.

The following outputs were saved for use in subsequent analysis:

- Cross-validation performance metrics
- Out-of-fold predictions and predicted probabilities
- Fold-level feature importance and selection results

These outputs provide the basis for Notebook 4, where model performance can be examined in greater detail using ROC curves, confusion matrices, and comparative visualizations.